In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/sentiment_analysis')
print(os.listdir())

Mounted at /content/drive
['train.csv', 'val.csv', 'test.csv', 'toxic-model']


In [5]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = "distilroberta-base"
labels = ["normal", "abusive", "threat"]
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def load(path):
    df = pd.read_csv(path).dropna(subset=["text"])
    df["label"] = df["category"].map(label2id)
    return Dataset.from_pandas(df[["text", "label"]], preserve_index=False)

train_ds = load("train.csv")
val_ds = load("val.csv")
test_ds = load("test.csv")
print(train_ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

Dataset({
    features: ['text', 'label'],
    num_rows: 89529
})


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
print(train_ds.features)

{'text': Value('string'), 'label': Value('int64')}


In [7]:
print(train_ds[:5])

{'text': ['Afghanistan is NOT a part of greater india, but of greater iran Whoever said afghanistan is part of greater india is an idiot. The hindu kush lies in pakistan as well as afghanistan, and the borders of india was set by the pakistani side. Afghans are white, Indians are Brown.', 'Stop reverting my edits Please stop reverting my edits, you online stalker. Who do you think you are?', 'I have also taken away the warning about factuality of the article. Although I still have qualms about the groups initial links to al-Qaeda I left them in the article becase there could be some legitimacy behind them (I cant read thh aljazeerah article) keep editing this if you come accross more information.', 'Dude, dude fuck you!!!', '" Support for that any many more similar statements is ALL OVER the Iranian media, including the ones who are not in line with Ahmadinejad from a political or ideological point of view. Playing the retarded trick of ""bring me a verifiable quote"" for every sentenc

In [8]:
print(train_ds[1])

{'text': 'Stop reverting my edits Please stop reverting my edits, you online stalker. Who do you think you are?', 'label': 1}


In [9]:
sample = "I WILL find you, idiooot"
print(tokenizer.tokenize(sample))
print(tokenizer(sample)["input_ids"])

['I', 'ĠWILL', 'Ġfind', 'Ġyou', ',', 'Ġid', 'io', 'oot']
[0, 100, 18342, 465, 47, 6, 13561, 1020, 9210, 2]


In [10]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
print(train_ds)

Map:   0%|          | 0/89529 [00:00<?, ? examples/s]

Map:   0%|          | 0/26787 [00:00<?, ? examples/s]

Map:   0%|          | 0/26787 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 89529
})


In [11]:
print(train_ds["input_ids"])

Column([[0, 15426, 5158, 7566, 16, 8127, 10, 233, 9, 2388, 9473, 493, 6, 53, 9, 2388, 10209, 260, 32396, 26, 9724, 5158, 7566, 16, 233, 9, 2388, 9473, 493, 16, 41, 30603, 4, 20, 1368, 36232, 449, 3810, 5738, 11, 181, 677, 7566, 25, 157, 25, 9724, 5158, 7566, 6, 8, 5, 7562, 9, 9473, 493, 21, 278, 30, 5, 181, 677, 41364, 526, 4, 33468, 32, 1104, 6, 6739, 32, 1547, 4, 2], [0, 22174, 41480, 2577, 127, 39708, 3401, 912, 41480, 2577, 127, 39708, 6, 47, 804, 1690, 38198, 4, 3394, 109, 47, 206, 47, 32, 116, 2], [0, 100, 33, 67, 551, 409, 5, 2892, 59, 754, 39416, 9, 5, 1566, 4, 2223, 38, 202, 33, 22043, 4339, 59, 5, 1134, 2557, 5678, 7, 1076, 12, 25697, 38, 314, 106, 11, 5, 1566, 44949, 3175, 89, 115, 28, 103, 19692, 639, 106, 36, 100, 17672, 1166, 3553, 298, 1076, 267, 10129, 254, 895, 1566, 43, 489, 5390, 42, 114, 47, 283, 7678, 14500, 55, 335, 4, 2], [0, 495, 6343, 6, 22633, 26536, 47, 16506, 2], [0, 113, 7737, 13, 14, 143, 171, 55, 1122, 1997, 16, 12389, 25133, 5, 5051, 433, 6, 217, 5, 1980

In [12]:
print(train_ds[:5].keys())

for i in range(5):
    print(train_ds[i])

dict_keys(['text', 'label', 'input_ids', 'attention_mask'])
{'text': 'Afghanistan is NOT a part of greater india, but of greater iran Whoever said afghanistan is part of greater india is an idiot. The hindu kush lies in pakistan as well as afghanistan, and the borders of india was set by the pakistani side. Afghans are white, Indians are Brown.', 'label': 0, 'input_ids': [0, 15426, 5158, 7566, 16, 8127, 10, 233, 9, 2388, 9473, 493, 6, 53, 9, 2388, 10209, 260, 32396, 26, 9724, 5158, 7566, 16, 233, 9, 2388, 9473, 493, 16, 41, 30603, 4, 20, 1368, 36232, 449, 3810, 5738, 11, 181, 677, 7566, 25, 157, 25, 9724, 5158, 7566, 6, 8, 5, 7562, 9, 9473, 493, 21, 278, 30, 5, 181, 677, 41364, 526, 4, 33468, 32, 1104, 6, 6739, 32, 1547, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
{'text': 'Stop rever

In [13]:
print(train_ds.select_columns(
    ["text", "label", "input_ids", "attention_mask"]
)[:5])

{'text': ['Afghanistan is NOT a part of greater india, but of greater iran Whoever said afghanistan is part of greater india is an idiot. The hindu kush lies in pakistan as well as afghanistan, and the borders of india was set by the pakistani side. Afghans are white, Indians are Brown.', 'Stop reverting my edits Please stop reverting my edits, you online stalker. Who do you think you are?', 'I have also taken away the warning about factuality of the article. Although I still have qualms about the groups initial links to al-Qaeda I left them in the article becase there could be some legitimacy behind them (I cant read thh aljazeerah article) keep editing this if you come accross more information.', 'Dude, dude fuck you!!!', '" Support for that any many more similar statements is ALL OVER the Iranian media, including the ones who are not in line with Ahmadinejad from a political or ideological point of view. Playing the retarded trick of ""bring me a verifiable quote"" for every sentenc

In [14]:
lengths = [len(x) for x in train_ds["input_ids"]]
print("cut off:", sum(l == 128 for l in lengths) / len(lengths))

cut off: 0.15814987322543533


In [15]:
import numpy as np

lengths = np.array([len(x) for x in train_ds["input_ids"]])
labels_arr = np.array(train_ds["label"])

for i, name in enumerate(labels):
    mask = labels_arr == i
    print(f"{name:8} cut off: {(lengths[mask] == 128).mean():.3f}")

normal   cut off: 0.185
abusive  cut off: 0.113
threat   cut off: 0.046


In [16]:
train_ds, val_ds, test_ds = load("train.csv"), load("val.csv"), load("test.csv")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

lengths = [len(x) for x in train_ds["input_ids"]]
print("cut off:", sum(l == 256 for l in lengths) / len(lengths))

Map:   0%|          | 0/89529 [00:00<?, ? examples/s]

Map:   0%|          | 0/26787 [00:00<?, ? examples/s]

Map:   0%|          | 0/26787 [00:00<?, ? examples/s]

cut off: 0.04868813457092115


In [17]:
import numpy as np
import torch
import torch.nn as nn
from transformers import Trainer
from sklearn.metrics import f1_score, recall_score, precision_score

In [18]:

# Class weights from training counts
counts = np.bincount(train_ds["label"], minlength=3)
weights = torch.tensor(len(train_ds) / (3 * counts), dtype=torch.float)
print(dict(zip(labels, weights.numpy().round(2))))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_t = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels_t)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = logits.argmax(axis=-1)
    t = label2id["threat"]
    return {
        "macro_f1": f1_score(y, preds, average="macro"),
        "threat_recall": recall_score(y == t, preds == t),
        "threat_precision": precision_score(y == t, preds == t, zero_division=0),
    }

{'normal': np.float32(0.5), 'abusive': np.float32(1.17), 'threat': np.float32(7.23)}


In [19]:
import torch
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          DataCollatorWithPadding)

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="toxic-model",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,

    warmup_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1,Threat Recall,Threat Precision
1,0.286819,0.310304,0.729647,0.943798,0.487976
2,0.237502,0.247807,0.811643,0.932171,0.608861
3,0.194608,0.259273,0.816721,0.926357,0.642473


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8394, training_loss=0.2583231222694875, metrics={'train_runtime': 1649.2557, 'train_samples_per_second': 162.853, 'train_steps_per_second': 5.09, 'total_flos': 1.7114488731016128e+16, 'train_loss': 0.2583231222694875, 'epoch': 3.0})

In [21]:
trainer.save_model("toxic-model-final")
tokenizer.save_pretrained("toxic-model-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('toxic-model-final/tokenizer_config.json', 'toxic-model-final/tokenizer.json')

In [22]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import confusion_matrix, classification_report

val_out = trainer.predict(val_ds)
val_proba = softmax(val_out.predictions, axis=1)
val_preds = val_proba.argmax(axis=1)

print(classification_report(val_out.label_ids, val_preds, target_names=labels, digits=3))
print(confusion_matrix(val_out.label_ids, val_preds))

              precision    recall  f1-score   support

      normal      0.993     0.916     0.953     23096
     abusive      0.618     0.918     0.739      3175
      threat      0.642     0.926     0.759       516

    accuracy                          0.917     26787
   macro avg      0.751     0.920     0.817     26787
weighted avg      0.941     0.917     0.924     26787

[[21163  1780   153]
 [  146  2916   113]
 [   12    26   478]]


In [23]:
from sklearn.metrics import recall_score, precision_score, f1_score

t_idx = label2id["threat"]
y_true = val_out.label_ids == t_idx

for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred = val_proba[:, t_idx] >= t
    print(f"threshold {t:.1f} | recall {recall_score(y_true, y_pred):.3f} "
          f"| precision {precision_score(y_true, y_pred):.3f} "
          f"| F1 {f1_score(y_true, y_pred):.3f}")

threshold 0.2 | recall 0.936 | precision 0.601 | F1 0.732
threshold 0.3 | recall 0.932 | precision 0.622 | F1 0.746
threshold 0.4 | recall 0.930 | precision 0.638 | F1 0.757
threshold 0.5 | recall 0.926 | precision 0.645 | F1 0.761
threshold 0.6 | recall 0.924 | precision 0.665 | F1 0.774
threshold 0.7 | recall 0.917 | precision 0.686 | F1 0.784
threshold 0.8 | recall 0.903 | precision 0.702 | F1 0.790
threshold 0.9 | recall 0.882 | precision 0.728 | F1 0.798


In [24]:
p_normal = val_proba[:, label2id["normal"]]
flag_true = val_out.label_ids != label2id["normal"]

for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    flag_pred = (1 - p_normal) >= t
    print(f"flag threshold {t:.1f} | harmful recall {recall_score(flag_true, flag_pred):.3f} "
          f"| precision {precision_score(flag_true, flag_pred):.3f} "
          f"| normal wrongly flagged {(flag_pred & ~flag_true).sum()}")

flag threshold 0.5 | harmful recall 0.958 | precision 0.646 | normal wrongly flagged 1941
flag threshold 0.6 | harmful recall 0.952 | precision 0.666 | normal wrongly flagged 1761
flag threshold 0.7 | harmful recall 0.942 | precision 0.688 | normal wrongly flagged 1577
flag threshold 0.8 | harmful recall 0.928 | precision 0.717 | normal wrongly flagged 1354
flag threshold 0.9 | harmful recall 0.894 | precision 0.766 | normal wrongly flagged 1008


In [25]:
val_df = pd.read_csv("val.csv").dropna(subset=["text"]).reset_index(drop=True)
mask = (val_out.label_ids == 0) & (val_preds == 1)
for text in val_df.loc[mask, "text"].sample(15, random_state=1):
    print("-", text[:200])

- " Yet again, you have demeaned another contributor. Individuals endowed with a smidgeon of true intuitive ability would easily recognize the reverent nature Hilarleo’s request. Unfortunately the comme
- Yo mumma Yo mumma Yo mumma Yo mumma Yo mumma Yo mumma Yo mumma Yo mumma Yo mumma Yo mumma Yo mumma Yo mummaYo mumma Yo mumma Yo mumma Yo mumma
- You can cut it out with the spam. I only made one violation, get off your high horse.
- " ==Mentioned in CYE== * It is stated in Larry David: Curb Your Enthusiasm that "" ... Molly Shannon fucked two guys to death"". The first was her fiance business manager during her first year at SNL 
- If you think that the people opposing these developers aren't also doing it out of greed then you're delusional. Same as most of the people who want to ban any building into the Green Belt... let's do
- hahah to the whole lot i say - it sure degrades the idea of rfa when you read all the crap above
- Ooooo, I love it when you talk dirty to me.
- never being

In [26]:
FLAG_T = 0.7
THREAT_T = 0.7

def decide(proba):
    p_normal = proba[:, label2id["normal"]]
    p_abusive = proba[:, label2id["abusive"]]
    p_threat = proba[:, label2id["threat"]]

    preds = np.full(len(proba), label2id["normal"])
    flagged = (1 - p_normal) >= FLAG_T
    preds[flagged] = label2id["abusive"]
    preds[flagged & (p_threat >= THREAT_T)] = label2id["threat"]
    return preds

# Confirm on validation first
val_final = decide(val_proba)
print(classification_report(val_out.label_ids, val_final, target_names=labels, digits=3))

              precision    recall  f1-score   support

      normal      0.990     0.932     0.960     23096
     abusive      0.660     0.907     0.764      3175
      threat      0.686     0.917     0.784       516

    accuracy                          0.929     26787
   macro avg      0.778     0.918     0.836     26787
weighted avg      0.945     0.929     0.933     26787



In [27]:
test_out = trainer.predict(test_ds)
test_proba = softmax(test_out.predictions, axis=1)
test_final = decide(test_proba)

print(classification_report(test_out.label_ids, test_final, target_names=labels, digits=3))
print(confusion_matrix(test_out.label_ids, test_final))

              precision    recall  f1-score   support

      normal      0.989     0.930     0.959     23096
     abusive      0.651     0.900     0.756      3176
      threat      0.681     0.901     0.776       515

    accuracy                          0.926     26787
   macro avg      0.774     0.910     0.830     26787
weighted avg      0.943     0.926     0.931     26787

[[21490  1491   115]
 [  217  2857   102]
 [   12    39   464]]


In [28]:
import torch, html, nltk
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import sent_tokenize


In [29]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

tok = AutoTokenizer.from_pretrained("toxic-model-final")
mdl = AutoModelForSequenceClassification.from_pretrained("toxic-model-final").eval()

def score(sentences):
    enc = tok(sentences, truncation=True, max_length=128,
              padding=True, return_tensors="pt")
    with torch.no_grad():
        logits = mdl(**enc).logits
    return torch.softmax(logits, dim=1).numpy()

COLORS = {"threat": "#ff9b9b", "abusive": "#ffd59b"}

def analyze_email(text):
    sentences = sent_tokenize(text)
    preds = decide(score(sentences))
    names = [labels[p] for p in preds]

    if "threat" in names:
        email_label = "threat"
    elif "abusive" in names:
        email_label = "abusive"
    else:
        email_label = "normal"

    parts = []
    for s, n in zip(sentences, names):
        s = html.escape(s)
        parts.append(f'<mark style="background:{COLORS[n]}">{s}</mark>' if n in COLORS else s)
    return email_label, " ".join(parts)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [30]:
email = """Hi team, please find the quarterly report attached.
Sorry to hear about your grandfather passing away last week.
If you report me to HR again, I will make sure you regret it.
Also, you are a useless idiot. Regards, Mark"""

label, highlighted = analyze_email(email)
print("Email label:", label)

with open("result.html", "w", encoding="utf-8") as f:
    f.write(f"<h3>Label: {label}</h3><p>{highlighted}</p>")

Email label: abusive


In [3]:
import os
print(os.getcwd())
print(os.path.abspath("toxic-model-final"))
print(os.path.exists("toxic-model-final"))

/content
/content/toxic-model-final
False


In [5]:
print("trainer" in globals())

False


In [6]:
from google.colab import drive
drive.mount('/content/drive')

!find /content -maxdepth 6 -name "toxic-model*" 2>/dev/null
!find /content -maxdepth 6 -name "checkpoint-*" 2>/dev/null

Mounted at /content/drive
/content/drive/MyDrive/sentiment_analysis/toxic-model
/content/drive/.Encrypted/MyDrive/sentiment_analysis/toxic-model
/content/drive/MyDrive/sentiment_analysis/toxic-model/checkpoint-8394
/content/drive/MyDrive/sentiment_analysis/toxic-model/checkpoint-2798
/content/drive/MyDrive/sentiment_analysis/toxic-model/checkpoint-5596
/content/drive/.Encrypted/MyDrive/sentiment_analysis/toxic-model/checkpoint-8394
/content/drive/.Encrypted/MyDrive/sentiment_analysis/toxic-model/checkpoint-2798
/content/drive/.Encrypted/MyDrive/sentiment_analysis/toxic-model/checkpoint-5596


In [7]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.chdir("/content/drive/MyDrive/sentiment_analysis")

ckpt = "toxic-model/checkpoint-8394"
model = AutoModelForSequenceClassification.from_pretrained(ckpt)
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

model.save_pretrained("toxic-model-final")
tokenizer.save_pretrained("toxic-model-final")
print(os.listdir("toxic-model-final"))
print(model.config.id2label)

Loading weights:   0%|          | 0/105 [00:02<?, ?it/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']
{0: 'normal', 1: 'abusive', 2: 'threat'}


In [8]:
import torch
tests = ["Thanks for the update, see you Monday.",
         "You are a pathetic idiot.",
         "I will find you and kill you."]
enc = tokenizer(tests, padding=True, return_tensors="pt")
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=1)
for t, p in zip(tests, probs):
    print(model.config.id2label[p.argmax().item()], "|", t)

normal | Thanks for the update, see you Monday.
abusive | You are a pathetic idiot.
threat | I will find you and kill you.
